<a href="https://colab.research.google.com/github/Jeremie-pires/big-data-projet/blob/main/Projet_Big_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modèle de Prédiction de Victimisatons Criminelles

Ce carnet Jupyter développe un modèle d'apprentissage automatique pour prédire les victimisations criminelles basé sur diverses caractéristiques démographiques et géographiques. Il couvre le chargement des données, le prétraitement, l'entraînement du modèle, son évaluation et l'optimisation des hyperparamètres.

### Chargement des données, nettoyage initial et ingénierie des fonctionnalités

Cette section charge l'ensemble de données `crimes.csv`, effectue un nettoyage initial en supprimant les espaces superflus des noms de colonnes et gère les valeurs manquantes dans la variable cible (`Victimisations_calendar_year_2015`).

Elle réalise également une ingénierie de fonctionnalités de base en créant `Log_Population` (population transformée par logarithme) et `Is_Auckland` (un indicateur binaire pour la région d'Auckland). Enfin, elle divise les données en ensembles d'entraînement et de test pour le développement et l'évaluation du modèle.

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

In [9]:
df = pd.read_csv('crimes.csv')
df.columns = df.columns.str.strip()

# Nettoyage et Ingénierie de variables
df = df.dropna(subset=['Victimisations_calendar_year_2015'])
df['Log_Population'] = np.log1p(df['Population_mid_point_2015'])
df['Is_Auckland'] = df['Region_2013_label'].apply(lambda x: 1 if 'Auckland' in str(x) else 0)

# Sélection des caractéristiques
exclude = ['Victimisations_calendar_year_2015', 'Rate_per_10000_population', 'Rate_ratio_NZ_average_rate',
           'Index', 'Area_unit_2013_code', 'Area_unit_2013_label', 'Urban_area_2013_code',
           'Territorial_authority_area_2013_code', 'Region_2013_code']

X = df.drop(columns=[col for col in exclude if col in df.columns])
y = df['Victimisations_calendar_year_2015']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Données préparées : {X_train.shape[0]} lignes d'entraînement, {X_test.shape[0]} lignes de test.")

✅ Données préparées : 1616 lignes d'entraînement, 404 lignes de test.


### Définition du pipeline de prétraitement des données

Ici, un `ColumnTransformer` est utilisé pour définir des étapes de prétraitement distinctes pour les caractéristiques numériques et catégorielles.

*   **Caractéristiques numériques** : Les valeurs manquantes sont imputées avec la médiane (`SimpleImputer`), puis mises à l'échelle à l'aide de `StandardScaler`.
*   **Caractéristiques catégorielles** : Les valeurs manquantes sont imputées avec la valeur la plus fréquente (`SimpleImputer`), puis encodées en 'one-hot' (`OneHotEncoder`).

Ces étapes de prétraitement sont intégrées dans un `Pipeline` avec un modèle de `LinearRegression` pour servir de référence.

In [10]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

# Pipeline de prétraitement
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

# Baseline (Régression Linéaire)
pipeline_baseline = Pipeline(steps=[('preprocessor', preprocessor), ('model', LinearRegression())])

print(f"✅ Pipeline de prétraitement configuré pour {len(numeric_features)} variables numériques et {len(categorical_features)} catégorielles.")

✅ Pipeline de prétraitement configuré pour 3 variables numériques et 4 catégorielles.


### Entraînement et évaluation du modèle de référence

Cette section entraîne le `pipeline_baseline` (qui comprend le prétraitement et un modèle de régression linéaire) sur les données d'entraînement. Après l'entraînement, il prédit les victimisations sur l'ensemble de test et évalue les performances du modèle à l'aide du R-squared (R²) et de l'erreur absolue moyenne (MAE).

In [11]:
pipeline_baseline.fit(X_train, y_train)
y_pred = pipeline_baseline.predict(X_test)

print("=== PERFORMANCES BASELINE ===")
print(f"R² : {r2_score(y_test, y_pred):.4f}")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f} agressions")

=== PERFORMANCES BASELINE ===
R² : 0.0774
MAE : 9.69 agressions


### Comparaison de modèles avec validation croisée

Cette section approfondit le modèle de référence en introduisant deux modèles de régression plus avancés : `RandomForestRegressor` et `GradientBoostingRegressor`.

Elle utilise une validation croisée à 5 plis sur l'ensemble d'entraînement pour comparer les performances (score R²) des trois modèles (régression linéaire, forêt aléatoire et 'Gradient Boosting'). Cela permet d'identifier le modèle le plus prometteur avant l'optimisation des hyperparamètres.

In [12]:
modeles = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

print("=== COMPARAISON PAR VALIDATION CROISÉE ===")
for nom, modele in modeles.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', modele)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    print(f"{nom:20} | R² Moyen: {scores.mean():.4f}")

=== COMPARAISON PAR VALIDATION CROISÉE ===
Linear Regression    | R² Moyen: 0.0989
Random Forest        | R² Moyen: 0.0473
Gradient Boosting    | R² Moyen: 0.2441


### Optimisation des hyperparamètres et exportation du modèle final

Cette dernière section se concentre sur l'optimisation du modèle le plus performant de l'étape de validation croisée ('Gradient Boosting').

`GridSearchCV` est utilisé pour rechercher les hyperparamètres optimaux (`n_estimators`, `learning_rate`, `max_depth`) pour le `GradientBoostingRegressor`. Le meilleur modèle trouvé est ensuite évalué une dernière fois sur l'ensemble de test non vu pour rapporter son score R² final.

Enfin, le modèle entraîné et optimisé est enregistré dans un fichier nommé `modele_crimes.pkl` à l'aide de `joblib` pour une utilisation future, par exemple pour un déploiement dans un tableau de bord.

In [13]:
# Optimisation du meilleur modèle (Gradient Boosting)
pipeline_gb = Pipeline(steps=[('preprocessor', preprocessor), ('model', GradientBoostingRegressor(random_state=42))])

param_grid = {
    'model__n_estimators': [50, 100, 150],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__max_depth': [3, 5]
}

grid_search = GridSearchCV(pipeline_gb, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Évaluation finale et export
best_model = grid_search.best_estimator_
final_r2 = r2_score(y_test, best_model.predict(X_test))
joblib.dump(best_model, 'modele_crimes.pkl')

print(f"✅ Modèle optimisé exporté ! R² Final : {final_r2:.4f}")
print(f"Meilleurs paramètres : {grid_search.best_params_}")

✅ Modèle optimisé exporté ! R² Final : 0.0730
Meilleurs paramètres : {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 50}


### Optimisation des hyperparamètres et exportation du modèle final

Cette dernière section se concentre sur l'optimisation du modèle le plus performant de l'étape de validation croisée ('Gradient Boosting').

`GridSearchCV` est utilisé pour rechercher les hyperparamètres optimaux (`n_estimators`, `learning_rate`, `max_depth`) pour le `GradientBoostingRegressor`. Le meilleur modèle trouvé est ensuite évalué une dernière fois sur l'ensemble de test non vu pour rapporter son score R² final.

Enfin, le modèle entraîné et optimisé est enregistré dans un fichier nommé `modele_crimes.pkl` à l'aide de `joblib` pour une utilisation future, par exemple pour un déploiement dans un tableau de bord.